# Video stage: Wan 2.2 TI2V-5B image-to-video clips

Animates chosen keyframes into short clips. Reads the storyboard and the keyframes from the public GitHub repo, so **push before running**.

* Each clip is cached by a hash of its inputs (keyframe hash, motion prompt, size, frames, steps, seed): a shot is regenerated only if one of them changed.
* Clips are saved to Google Drive **as they finish**, so a disconnect loses at most the clip in progress.
* Every call is logged in the same format as `runs.jsonl`, with GPU minutes.

**Run:** Runtime → Change runtime type → **A100 GPU** → Save, then Runtime → Run all. Allow Google Drive access when asked.

**After:** unzip `clips.zip` into `outputs/clips/` locally, append `clip_runs.jsonl` to `runs.jsonl`, run `make assemble`.

In [ ]:
!pip -q install -U "diffusers>=0.35" transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
# ---- Settings (change these, not the code below) ----
SHOTS = ["s07", "s03", "s09"]        # in priority order: most likely to succeed first
WIDTH, HEIGHT = 704, 1280            # 720p portrait, the model's native size (480x832 is ~2x faster)
FRAMES = 121                         # 121 frames = 5 s at 24 fps
STEPS = 40                           # 50 is the default; 40 saves ~20% time
GUIDANCE = 5.0
STYLE = "hand-painted gouache illustration, gentle natural motion, stable faces and hands"
NEGATIVE = ("blurry, distorted, deformed hands, extra fingers, morphing face, identity change, "
            "flicker, jitter, text, watermark, low quality, photorealistic")

import json, os, hashlib, urllib.request
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/heritage-short/clips"
os.makedirs(OUT, exist_ok=True)

RAW = "https://raw.githubusercontent.com/Robelabcd/heritage-short/main/"
def fetch_json(path, default=None):
    try:
        return json.load(urllib.request.urlopen(RAW + path))
    except Exception:
        return default

storyboard = fetch_json("storyboard.json")
kf_manifest = fetch_json("outputs/keyframes/manifest.json")
shots = {s["id"]: s for s in storyboard["shots"]}
manifest_path = f"{OUT}/manifest.json"
manifest = (json.load(open(manifest_path)) if os.path.exists(manifest_path)
            else fetch_json("outputs/clips/manifest.json", {}))

def input_hash(inputs):
    return hashlib.sha256(json.dumps(inputs, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]

for sid in SHOTS:
    print(sid, "|", shots[sid]["motion_prompt"])

In [ ]:
import torch, time
from diffusers import WanImageToVideoPipeline, AutoencoderKLWan

MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
gpu = torch.cuda.get_device_name(0)
t0 = time.time()
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.bfloat16)
if torch.cuda.get_device_properties(0).total_memory > 70e9:
    pipe.to("cuda")                      # 80 GB cards: everything on the GPU
else:
    pipe.enable_model_cpu_offload()      # 40 GB and smaller: move parts to the GPU only when needed
pipe.vae.enable_tiling()                 # decode the video in tiles to save memory
print(f"loaded {MODEL} on {gpu} in {time.time() - t0:.0f}s")

In [ ]:
from PIL import Image
from io import BytesIO
from datetime import datetime, timezone
from diffusers.utils import export_to_video
from IPython.display import Video, display

def keyframe(sid):
    name = next(n for n in kf_manifest if n.rsplit(".", 1)[0] == sid)
    img = Image.open(BytesIO(urllib.request.urlopen(RAW + "outputs/keyframes/" + name).read())).convert("RGB")
    target = WIDTH / HEIGHT                                   # centre-crop to the model's aspect, then resize
    w, h = img.size
    if w / h > target:
        nw = int(h * target); img = img.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
    else:
        nh = int(w / target); img = img.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    return name, img.resize((WIDTH, HEIGHT), Image.LANCZOS)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
log_path = f"{OUT}/clip_runs.jsonl"
for sid in SHOTS:
    shot = shots[sid]
    kf_name, image = keyframe(sid)
    prompt = f"{shot['motion_prompt']}. {STYLE}"
    params = {"prompt": prompt, "negative_prompt": NEGATIVE, "width": WIDTH, "height": HEIGHT,
              "num_frames": FRAMES, "num_inference_steps": STEPS, "guidance_scale": GUIDANCE}
    digest = input_hash({"model": MODEL, "keyframe": kf_manifest[kf_name], **params, "seed": shot["seed"]})
    name = f"{sid}.mp4"
    if manifest.get(name) == digest and os.path.exists(f"{OUT}/{name}"):
        print(f"cached  {name}"); continue
    torch.cuda.synchronize(); t = time.time()
    status, error = "ok", None
    try:
        frames = pipe(image=image, **params,
                      generator=torch.Generator("cuda").manual_seed(shot["seed"])).frames[0]
        export_to_video(frames, f"{OUT}/{name}", fps=24)
        manifest[name] = digest
        json.dump(manifest, open(manifest_path, "w"), indent=2, sort_keys=True)
    except Exception as exc:                                  # log the failure and move on to the next shot
        status, error = "error", f"{type(exc).__name__}: {exc}"[:500]
        torch.cuda.empty_cache()
    latency = round(time.time() - t, 3)
    with open(log_path, "a") as f:
        f.write(json.dumps({"ts": datetime.now(timezone.utc).isoformat(timespec="seconds"), "run_id": run_id,
                            "stage": "clips", "shot": sid, "provider": f"colab ({gpu})", "model": MODEL,
                            "params": params, "seed": shot["seed"], "attempt": 1, "status": status,
                            "latency_s": latency, "gpu_minutes": round(latency / 60, 2), "cost_usd": 0.0,
                            **({"error": error} if error else {})}) + "\n")
    print(f"{status:6}  {name}  {latency / 60:.1f} min" + (f"  {error}" if error else ""))
    if status == "ok":
        display(Video(f"{OUT}/{name}", embed=True, width=240))

In [ ]:
!cd "/content/drive/MyDrive/heritage-short/clips" && zip -q -o /content/clips.zip *.mp4 manifest.json clip_runs.jsonl
from google.colab import files
files.download("/content/clips.zip")